In [0]:
%sql
-- DIM_DEPARTMENT
-- Lowest-grain organisational dimension; other dims reference department_id
-- as a natural attribute (not a surrogate FK), since the business requirement
-- only asks for surrogate-key substitution inside the fact table.
-- ----------------------------------------------------------------------------
CREATE OR REPLACE TABLE university_bot.analytics.dim_department (
    department_key   BIGINT GENERATED ALWAYS AS IDENTITY,   -- surrogate key
    department_id    BIGINT NOT NULL,                        -- business/natural key
    department_name  STRING,
    faculty_id       BIGINT,
    created_at       TIMESTAMP,
    updated_at       TIMESTAMP
)
USING DELTA
COMMENT 'Department dimension - one row per academic department';

-- ----------------------------------------------------------------------------
-- DIM_PROFESSOR
-- ----------------------------------------------------------------------------
CREATE OR REPLACE TABLE university_bot.analytics.dim_professor (
    professor_key     BIGINT GENERATED ALWAYS AS IDENTITY,
    professor_id      BIGINT NOT NULL,
    professor_name    STRING,
    professor_email   STRING,
    academic_title    STRING,
    department_id     BIGINT,
    created_at        TIMESTAMP,
    updated_at        TIMESTAMP
)
USING DELTA
COMMENT 'Professor dimension - one row per professor';

-- ----------------------------------------------------------------------------
-- DIM_COURSE
-- ----------------------------------------------------------------------------
CREATE OR REPLACE TABLE university_bot.analytics.dim_course (
    course_key       BIGINT GENERATED ALWAYS AS IDENTITY,
    course_id        BIGINT NOT NULL,
    course_code      STRING,
    course_name      STRING,
    credit_hours     INT,
    study_year       STRING,
    semester         STRING,
    department_id    BIGINT,
    created_at       TIMESTAMP,
    updated_at       TIMESTAMP
)
USING DELTA
COMMENT 'Course dimension - one row per course offering definition';

-- ----------------------------------------------------------------------------
-- DIM_STUDENT
-- ----------------------------------------------------------------------------
CREATE OR REPLACE TABLE university_bot.analytics.dim_student (
    student_key         BIGINT GENERATED ALWAYS AS IDENTITY,
    student_id          BIGINT NOT NULL,
    national_id         STRING,
    student_name        STRING,
    phone               STRING,
    gender              STRING,
    birthdate           DATE,
    study_year          STRING,
    department_id       BIGINT,
    enrollment_year     STRING,
    telegram_username   STRING,
    created_at          TIMESTAMP,
    updated_at          TIMESTAMP
)
USING DELTA
COMMENT 'Student dimension - one row per student';

-- ----------------------------------------------------------------------------
-- DIM_SESSION
-- ----------------------------------------------------------------------------
CREATE OR REPLACE TABLE university_bot.analytics.dim_session (
    session_key      BIGINT GENERATED ALWAYS AS IDENTITY,
    session_id       BIGINT NOT NULL,
    course_id        BIGINT,
    professor_id     BIGINT,
    session_type     STRING,
    day_of_week      STRING,
    start_time       STRING,
    end_time         STRING,
    location         STRING,
    semester         STRING,
    academic_year    STRING,
    created_at       TIMESTAMP,
    updated_at       TIMESTAMP
)
USING DELTA
COMMENT 'Session dimension - one row per scheduled class session';


/* ============================================================================
   STEP 3: CREATE FACT TABLE
   Grain: one row per attendance event (student + session + date).
   Natural keys from the OLTP attendance table are resolved to surrogate
   keys during the load step below.
   ============================================================================ */
CREATE OR REPLACE TABLE university_bot.analytics.fact_attendance (
    attendance_key      BIGINT GENERATED ALWAYS AS IDENTITY,  -- surrogate key
    student_key         BIGINT NOT NULL,                       -- FK -> dim_student
    course_key          BIGINT NOT NULL,                       -- FK -> dim_course
    professor_key       BIGINT NOT NULL,                       -- FK -> dim_professor
    session_key         BIGINT NOT NULL,                       -- FK -> dim_session
    attendance_date     DATE,
    attendance_status   STRING,
    check_in_time       TIMESTAMP,
    created_at          TIMESTAMP
)
USING DELTA
COMMENT 'Fact table - one row per attendance event, ready for BI aggregation';


/* ============================================================================
   STEP 4: ADD PRIMARY KEY / FOREIGN KEY METADATA
   Informational constraints only (NOT ENFORCED). Enables auto-join
   detection in Databricks Catalog Explorer, Genie, and BI tools.
   ============================================================================ */

/*-- Dimension primary keys
ALTER TABLE university_bot.analytics.dim_department ADD CONSTRAINT pk_dim_department PRIMARY KEY (department_key);
ALTER TABLE university_bot.analytics.dim_professor  ADD CONSTRAINT pk_dim_professor  PRIMARY KEY (professor_key);
ALTER TABLE university_bot.analytics.dim_course     ADD CONSTRAINT pk_dim_course     PRIMARY KEY (course_key);
ALTER TABLE university_bot.analytics.dim_student    ADD CONSTRAINT pk_dim_student    PRIMARY KEY (student_key);
ALTER TABLE university_bot.analytics.dim_session    ADD CONSTRAINT pk_dim_session    PRIMARY KEY (session_key);

-- Fact primary key
ALTER TABLE university_bot.analytics.fact_attendance ADD CONSTRAINT pk_fact_attendance PRIMARY KEY (attendance_key);*/

-- Fact foreign keys -> dimensions (star schema relationships)
ALTER TABLE university_bot.analytics.fact_attendance ADD CONSTRAINT fk_fact_student
    FOREIGN KEY (student_key) REFERENCES dim_student(student_key);

ALTER TABLE university_bot.analytics.fact_attendance ADD CONSTRAINT fk_fact_course
    FOREIGN KEY (course_key) REFERENCES dim_course(course_key);

ALTER TABLE university_bot.analytics.fact_attendance ADD CONSTRAINT fk_fact_professor
    FOREIGN KEY (professor_key) REFERENCES dim_professor(professor_key);

ALTER TABLE university_bot.analytics.fact_attendance ADD CONSTRAINT fk_fact_session
    FOREIGN KEY (session_key) REFERENCES dim_session(session_key);

